# Bimanual affordance pipeline — step-by-step verification

Walk the generation pipeline one stage at a time, inspecting the **exact prompt sent** and the **raw model output** at every step.

**Pipeline**
1. **Object filtering** *(preserved)* — is this object bimanually operable?
2. **Component extraction** *(preserved)* — list groundable parts → `{object_id, object_name, components}`
3. **Task proposal** *(NEW)* — image-grounded daily tasks framed as *state changes*, with a bimanual-necessity check
4. **Coordination-aware role decomposition** *(NEW)* — task → two hand-agnostic roles + a free-form coordination relation
5. **Role pair pattern** *(NEW)* — deterministic normalization → `pattern` / `symmetric` / `hand_agnostic`

Run from the repo root with the `mm` conda env (e.g. `CUDA_VISIBLE_DEVICES=1`). Steps 3–5 reuse the **same** Gemma model loaded by `ComponentExtractor`, so the model is only loaded once.

> **Kernel:** pick **`Python (gemma4)`** (top-right) to run `gemma-4-12B-it` (transformers 5.x). The `mm` kernel would instead run `gemma-3-4b-it`. The model is chosen by the `GEMMA_MODEL_ID` env set in the next cell, so you can also override it there.

## 0 · Setup

In [ ]:
import os, sys, json, random
# --- pin GPU + select Gemma 4 (must be set before torch / model load) ---
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "3")
os.environ.setdefault("GEMMA_MODEL_ID", "google/gemma-4-12B-it")

import torch
from PIL import Image
from IPython.display import display

REPO_ROOT    = "/home/michaellee/mclee/affogato"
BIMANUAL_DIR = os.path.join(REPO_ROOT, "bimanual_annotation")
os.chdir(REPO_ROOT)                 # so relative paths (prompts, grid) resolve
sys.path.insert(0, BIMANUAL_DIR)    # so `from gemma import Gemma` resolves

MAPPING   = "data_generation/dataset/daily_used_to_affogato.json"
GRID_PATH = "tmp_grid.png"
OUT_PATH  = "data_generation/output_taskfirst_verify.json"

print("cwd:", os.getcwd(), "| GEMMA_MODEL_ID:", os.environ["GEMMA_MODEL_ID"])

In [ ]:
def parse_json(text):
    """Lenient: grab the outermost {...} and json.load it."""
    try:
        start = text.find("{"); end = text.rfind("}") + 1
        return json.loads(text[start:end])
    except Exception as e:
        return {"error": f"parse failed: {e}", "raw": text}

def components_to_text(components):
    """list[{name, interaction}] -> bullet lines for the prompt."""
    return "\n".join(f"- {c['name']}: {c.get('interaction', '')}" for c in components)

def show_and_run(user_text, system_text, model, image_url=None):
    """Print the exact prompt sent, run it, print + return the raw output."""
    print("=" * 72)
    # print("SYSTEM:\n" + system_text + "\n")
    # print("USER:\n" + user_text + "\n" + "-" * 72)
    raw = (model.text_image(user_text, image_url, system_text)
           if image_url else model.text(user_text, system_text))
    print("RAW OUTPUT:\n" + raw)
    print("=" * 72)
    return raw

## 1 · Pick an object

Tweak `IDX` to inspect a different object. (If `object_name` is `unknown object` or fewer than 4 views exist, try another `IDX`.)

In [ ]:
from get_component import load_object_name, sample_views, make_grid

data = [d for d in json.load(open(MAPPING)) if d.get("dst")]
print(f"{len(data)} mapped objects")

IDX = 66617   # <-- tweak to inspect a different object
ex  = data[IDX]

object_id   = ex["object_id"]
object_name = load_object_name(ex["dst"])
views       = sample_views(ex["src"], 4)
make_grid(views, GRID_PATH)

print("object_id  :", object_id)
print("object_name:", object_name)
print("views      :", len(views), " exist:", sum(os.path.exists(v) for v in views))
display(Image.open(GRID_PATH))

## 2 · Object filtering (redesigned) + component extraction

**Filtering is redesigned** — it now keeps an object when a two-handed task *exists*, instead of asking whether it is *designed* for two hands (which wrongly dropped boxes etc.). It rejects two things: objects that are weird / rarely handled in daily life, and objects with no plausible coordinated two-hand task. Output is a small JSON verdict so you can see *why* and the example task it had in mind.

**Component extraction is unchanged.** Together they emit the `object_record` — the "legible ID + json list" this stage produces.

In [ ]:
from get_component import ComponentExtractor

ce    = ComponentExtractor()   # loads Gemma (max_new_tokens=512)
model = ce.model               # reuse this same model for the NEW steps below

In [ ]:
# --- Step A1: object filtering (REDESIGNED, inline + iterable) ---
# Goal: drop (a) weird / rarely-handled objects, and (b) objects with NO realistic
# two-handed task. Test is EXISTENCE of a bimanual task, not "designed for two hands"
# (so a box is KEPT: you can carry/move it with two hands).
FILTER_PROMPT = {
    "system": "You screen objects for a bimanual (two-arm) robot manipulation dataset.",
    "user": (
        "Object category: {object_name}\n"
        "The image shows four views of one object in a 2x2 grid.\n\n"
        "Decide whether to KEEP this object. KEEP only if BOTH hold:\n"
        "1. EVERYDAY: it is a common physical object a person would actually handle in "
        "daily life (home, kitchen, office, tools, containers, movable furniture). "
        "REJECT fantasy/game characters, purely decorative or ornamental pieces, fixed "
        "architecture/fixtures, body parts, and weird/rare things you would seldom handle.\n"
        "2. BIMANUAL TASK EXISTS: there is at least ONE realistic task in which two hands "
        "would coordinate to operate or move it. Judge whether such a task EXISTS, not "
        "whether the object was 'designed' for two hands. Carrying / lifting / moving a "
        "large or heavy object counts (two hands needed to carry it), as do opening, "
        "folding, twisting, pulling apart, stabilize-and-actuate, etc.\n"
        "REJECT only if even with two hands there is no meaningful coordinated task "
        "(e.g. a tiny object one hand fully handles, or something you never manipulate).\n\n"
        "Return JSON only:\n"
        "{{\n"
        "  \"everyday_object\": true/false,\n"
        "  \"bimanual_task_exists\": true/false,\n"
        "  \"example_task\": \"one realistic two-handed task, or null\",\n"
        "  \"keep\": true/false\n"
        "}}"
    ),
}

filt_raw = show_and_run(
    FILTER_PROMPT["user"].format(object_name=object_name),
    FILTER_PROMPT["system"], model, image_url=GRID_PATH
)
filt = parse_json(filt_raw)
operable = bool(filt.get("keep"))
print("\nkeep:", operable,
      "| everyday:", filt.get("everyday_object"),
      "| task_exists:", filt.get("bimanual_task_exists"))
print("example_task:", filt.get("example_task"))

In [ ]:
# --- Step A2: component extraction (preserved) ---
pp = ce.prompts["parse"]
comp_raw = show_and_run(
    pp["user"].format(object_name=object_name), pp["system"], model, image_url=GRID_PATH
)
components = parse_json(comp_raw).get("canonical_components", [])

object_record = {
    "object_id":   object_id,
    "object_name": object_name,
    "views_used":  views,
    "components":  components,
}
print("\n--- object_record ---")
print(json.dumps(object_record, indent=2, ensure_ascii=False))

## 3 · Task proposal  *(NEW)*: two families -> ranked -> assembled set

Tasks are proposed in **two families**, then down-selected to a fixed set:
- **inter_object** - whole-object tasks: move / carry / reorient / flip the ENTIRE object with both hands.
- **intra_object** - part-level tasks: one hand holds the body, the other opens / twists / pulls a movable part.

- **B1 brainstorm** - up to `N_PER_FAMILY` (5) candidates in EACH family.
- **B2 rank + assemble** - the LLM validates + ranks each family **most-daily-used first**, then Python takes `INTER_QUOTA` (3) + `INTRA_QUOTA` (3) = `TOTAL` (6); if a family is short it **backfills from the other** (never fabricates beyond valid tasks).

Hard guards (both stages): two hands acting **together** (no handoffs), **self-contained** (no external surfaces/objects), state/pose/part change, only groundable parts. The constants are one-line tweaks. `{{ }}` are literal braces for `str.format`.

In [43]:
# --- Step B1: brainstorm candidates in two families (over-generate, image-grounded) ---
N_PER_FAMILY = 8   # brainstorm up to this many per family; Step B2 down-selects to TOTAL

TASK_BRAINSTORM_PROMPT = {
    "system": (
        "You are a task planner for a two-arm (bimanual) robot. You brainstorm daily tasks "
        "the robot could do using ONLY its two hands and the object itself - never relying on "
        "walls, surfaces, other objects, or tools."
    ),
    "user": (
        "Object: {object_name}\n"
        "The image shows four views of this object in a 2x2 grid.\n"
        "Groundable parts (the robot can only contact these named parts):\n"
        "{components}\n\n"
        "Brainstorm daily tasks a two-arm robot could do with THIS object, in TWO families:\n"
        "A) inter_object - WHOLE-OBJECT tasks: treat the object as one rigid unit and move it with "
        "both hands together (move / carry / lift / reorient / flip / rotate the ENTIRE object).\n"
        "B) intra_object - PART-LEVEL tasks: operate the object's own parts - one hand holds the body "
        "while the other actuates a movable part (open/close, twist, pull, press, slide a part).\n\n"
        "Give up to {n} tasks in EACH family. Every task must be a COMMON, everyday operation "
        "people genuinely do with this object - keep it SIMPLE, DIRECT, and natural, and do NOT "
        "invent unusual, contrived, or rare tasks just to reach {n} or to look diverse; if only "
        "a few common operations exist, give only those.\n"
        "DIVERSITY means genuinely different ACTIONS / concepts (e.g. carry vs flip vs open). It "
        "is NOT a mirror variant: a 'left' vs 'right' version - or any task differing only by "
        "which symmetric side/part is used - is the SAME concept, so do not list both. (Reworded "
        "phrasings of the same task are still fine.)\n"
        "Each task MUST:\n"
        "- cause an object state / pose / part change (not \"look at\" / \"observe\" / \"sit on\")\n"
        "- need TWO hands acting AT THE SAME TIME in complementary roles (NOT hand-to-hand handoffs / "
        "passing / regrasps; co-carrying with both hands is fine)\n"
        "- be SELF-CONTAINED: two hands + this object only, no external surface/object/tool\n"
        "- only involve the groundable parts above (intra_object tasks must name real parts)\n\n"
        "Return JSON only:\n"
        "{{\n"
        "  \"inter_object\": [ {{\"task\": \"...\", \"goal\": \"<before> -> <after>\"}} ],\n"
        "  \"intra_object\": [ {{\"task\": \"...\", \"goal\": \"<before> -> <after>\"}} ]\n"
        "}}\n"
        "Output JSON only."
    ),
}

brainstorm_raw = show_and_run(
    TASK_BRAINSTORM_PROMPT["user"].format(
        object_name=object_name, components=components_to_text(components), n=N_PER_FAMILY
    ),
    TASK_BRAINSTORM_PROMPT["system"], model, image_url=GRID_PATH
)
bs = parse_json(brainstorm_raw)
cand_inter = bs.get("inter_object", [])
cand_intra = bs.get("intra_object", [])
print(f"\ninter_object ({len(cand_inter)}):")
for i, t in enumerate(cand_inter): print(f"  [{i}] {t.get('task')!r}  |  {t.get('goal')}")
print(f"intra_object ({len(cand_intra)}):")
for i, t in enumerate(cand_intra): print(f"  [{i}] {t.get('task')!r}  |  {t.get('goal')}")

RAW OUTPUT:
```json
{
  "inter_object": [
    {
      "task": "Lift the box",
      "goal": "on ground -> held in the air"
    },
    {
      "task": "Carry the box",
      "goal": "at point A -> at point B"
    },
    {
      "task": "Flip the box over",
      "goal": "top side up -> bottom side up"
    },
    {
      "task": "Rotate the box",
      "goal": "facing front -> facing side"
    },
    {
      "task": "Tilt the box",
      "goal": "level -> angled"
    }
  ],
  "intra_object": []
}
```

inter_object (5):
  [0] 'Lift the box'  |  on ground -> held in the air
  [1] 'Carry the box'  |  at point A -> at point B
  [2] 'Flip the box over'  |  top side up -> bottom side up
  [3] 'Rotate the box'  |  facing front -> facing side
  [4] 'Tilt the box'  |  level -> angled
intra_object (0):


In [44]:
# --- Step B2: rank each family "most daily-used first", then assemble the final set ---
TOTAL       = 5   # total queries per object
INTER_QUOTA = 3   # target picks from inter_object (whole-object)
INTRA_QUOTA = 3   # target picks from intra_object (part-level); shortfalls backfill across families

def candidates_to_text(cs):
    return "\n".join(f"- {c.get('task')} ({c.get('goal')})" for c in cs) or "- (none)"

TASK_RANK_PROMPT = {
    "system": (
        "You curate a two-arm (bimanual) robot manipulation dataset. You validate candidate tasks "
        "and rank them by how common/everyday they are (MOST daily-used first)."
    ),
    "user": (
        "Object: {object_name}\n"
        "Groundable parts:\n{components}\n\n"
        "inter_object candidates (whole-object):\n{inter}\n\n"
        "intra_object candidates (part-level):\n{intra}\n\n"
        "For EACH family: drop any task that breaks a rule, then RANK the survivors MOST "
        "DAILY-USED / most common FIRST. A valid task MUST:\n"
        "- need TWO hands acting AT THE SAME TIME in complementary roles (no handoffs / passing / "
        "regrasps; co-carrying is fine)\n"
        "- be SELF-CONTAINED (two hands + this object only; no walls/surfaces/other objects/tools)\n"
        "- cause an object state/pose/part change, and not be solvable by one hand alone\n"
        "- only involve the groundable parts above\n"
        "Keep tasks simple and natural; keep rewordings/similar tasks (only drop EXACT duplicates).\n\n"
        "Return JSON only (each family ranked, most-daily-used first):\n"
        "{{\n"
        "  \"inter_object\": [ {{\"task\": \"...\", \"goal\": \"<before> -> <after>\", "
        "\"why_bimanual\": \"what fails with one hand\"}} ],\n"
        "  \"intra_object\": [ {{\"task\": \"...\", \"goal\": \"<before> -> <after>\", "
        "\"why_bimanual\": \"what fails with one hand\"}} ]\n"
        "}}\n"
        "Output JSON only."
    ),
}

rank_raw = show_and_run(
    TASK_RANK_PROMPT["user"].format(
        object_name=object_name, components=components_to_text(components),
        inter=candidates_to_text(cand_inter), intra=candidates_to_text(cand_intra),
    ),
    TASK_RANK_PROMPT["system"], model, image_url=GRID_PATH
)
ranked = parse_json(rank_raw)
r_inter = [dict(t, category="inter") for t in ranked.get("inter_object", [])]
r_intra = [dict(t, category="intra") for t in ranked.get("intra_object", [])]

# deterministic assembly: quota per family, backfill across families, cap at TOTAL
picked = r_inter[:INTER_QUOTA] + r_intra[:INTRA_QUOTA]
if len(picked) < TOTAL:
    leftover = r_inter[INTER_QUOTA:] + r_intra[INTRA_QUOTA:]
    picked += leftover[:TOTAL - len(picked)]
tasks = picked[:TOTAL]

# attach the dataset task query (used directly as the final Task query)
QUERY_TEMPLATE = "How would you {task}?"
def to_query(task):
    t = str(task).strip().rstrip(".")
    if t:
        t = t[0].lower() + t[1:]
    return QUERY_TEMPLATE.format(task=t)
for t in tasks:
    t["query"] = to_query(t.get("task", ""))

n_inter = sum(t["category"] == "inter" for t in tasks)
print(f"\nassembled {len(tasks)}/{TOTAL} tasks  (inter={n_inter}, intra={len(tasks) - n_inter})")
for i, t in enumerate(tasks):
    print(f"  [{i}] ({t['category']}) {t.get('query')!r}")
    # print(f"       why: {t.get('why_bimanual')}")

RAW OUTPUT:
```json
{
  "inter_object": [
    {
      "task": "Carry the box",
      "goal": "at point A -> at point B",
      "why_bimanual": "A large box requires two hands to maintain stability and balance while moving."
    },
    {
      "task": "Flip the box over",
      "goal": "top side up -> bottom side up",
      "why_bimanual": "One hand cannot rotate a large box 180 degrees around its horizontal axis without it falling or slipping."
    },
    {
      "task": "Rotate the box",
      "goal": "facing front -> facing side",
      "why_bimanual": "Rotating a large, heavy box requires two hands to exert torque simultaneously to prevent it from tipping."
    },
    {
      "task": "Lift the box",
      "goal": "on ground -> held in the air",
      "why_bimanual": "A large box is too heavy or unstable to be lifted vertically by a single hand."
    },
    {
      "task": "Tilt the box",
      "goal": "level -> angled",
      "why_bimanual": "One hand cannot hold the base steady whi

## 4 · Coordination-aware role decomposition  *(NEW)*

For one chosen task, decompose into **exactly two hand-agnostic roles** (`A`, `B`) drawn from a fixed verb set, each bound to a groundable part, plus a coordination `relation`.

`relation` is **open / free-form** — the model writes its own short phrase for the force/motion dependency; `RELATION_EXAMPLES` only steers the granularity, it is not an allowed-list. Because the answer parts ARE the role targets, the old `reason` ↔ `affordance_parts` drift can't happen.

In [45]:
# --- Step C: coordination-aware role decomposition (NEW) ---
# Robot-executable, hand-agnostic role verbs (grounded core 9).
# See notes/role_vocabulary_grounding.md. Dropped vs old set: align (-> relation field),
# separate (-> two 'pull' + opposing relation), tilt (merged into axis-parameterized rotate).
ROLE_VERBS = ["hold", "support", "push", "pull", "press", "lift", "rotate", "slide", "insert"]
ROLE_GLOSS = (
    "hold = static prehensile grip that immobilizes a part (stabilizer); "
    "support = bear weight from below WITHOUT a grip (non-prehensile stabilizer); "
    "push = non-prehensile lateral force to translate; "
    "pull = grasp then retract along a line; "
    "press = localized normal force into a part that yields/clicks (button/latch/lid); "
    "lift = raise the whole object upward; "
    "rotate = axis-parameterized rotation - covers in-place twist of a knob/cap AND reorient/flip/tilt "
    "of the whole object; "
    "slide = translate a part along its linear/prismatic track; "
    "insert = mate a part into a receptacle/slot"
)
# relation is OPEN - these are only steering examples, NOT an allowed-list
RELATION_EXAMPLES = ["opposing torque", "stabilize and actuate", "balanced grip on opposite sides",
                     "opposing force", "orientation control", "guide and actuate",
                     "constrained translation"]

ROLE_DECOMP_PROMPT = {
    "system": (
        "You decompose a single bimanual task into two coordinated, hand-agnostic manipulation "
        "roles for a two-arm robot. A role is a robot-executable verb describing what ONE hand does."
    ),
    "user": (
        "Object: {object_name}\n"
        "Task: {task}\n"
        "Goal: {goal}\n"
        "Why bimanual: {why_bimanual}\n"
        "Groundable parts (a role's target MUST be one of these, copied verbatim):\n"
        "{components}\n\n"
        "Decompose this task into EXACTLY two coordinated roles, A and B.\n\n"
        "\"role\" MUST be one of: {role_verbs}\n"
        "Role meanings - {role_gloss}.\n"
        "\"target\" = the groundable PART each hand acts on (copied verbatim from the list).\n"
        "\"contact_region\" = a SHORT phrase naming WHERE on the object that hand actually contacts - "
        "this is the phrase the downstream grounder (Molmo) will point to.\n"
        "\"relation\" = a SHORT, free-form phrase describing the force/motion AND spatial dependency "
        "between the two roles (write your own; for calibration only, e.g.: {relation_examples}).\n\n"
        "Return JSON only:\n"
        "{{\n"
        "  \"roles\": [\n"
        "    {{\"id\": \"A\", \"role\": \"...\", \"target\": \"...\", \"contact_region\": \"where hand A grips\", \"function\": \"...\"}},\n"
        "    {{\"id\": \"B\", \"role\": \"...\", \"target\": \"...\", \"contact_region\": \"a DIFFERENT place from A\", \"function\": \"...\"}}\n"
        "  ],\n"
        "  \"relation\": \"...\"\n"
        "}}\n\n"
        "Rules:\n"
        "- Both roles must physically contact the object (no watch/observe/monitor).\n"
        "- The two contact_regions MUST denote two DIFFERENT physical locations - NEVER the same point "
        "(otherwise both hands would be sent to the same spot). This distinctness is part of the "
        "coordination.\n"
        "- For a SYMMETRIC whole-object task where both hands do the SAME verb (e.g. hold + hold to "
        "co-carry a box, or lift + lift), the two contact_regions MUST be OPPOSITE / complementary "
        "sides so the grip is balanced and distinct (e.g. 'left side of the box' vs 'right side of "
        "the box', or 'near end' vs 'far end').\n"
        "- For an ASYMMETRIC task the two roles act on different parts (e.g. hold body + rotate cap), "
        "so the contact_regions are naturally different.\n"
        "- target must be copied verbatim from the groundable parts.\n"
        "- role must come from the fixed list; relation is free-form but concise.\n"
        "- Output JSON only."
    ),
}

TASK_IDX = 0   # <-- which proposed task to decompose
chosen   = tasks[TASK_IDX]

decomp_raw = show_and_run(
    ROLE_DECOMP_PROMPT["user"].format(
        object_name=object_name,
        task=chosen.get("task", ""),
        goal=chosen.get("goal", ""),
        why_bimanual=chosen.get("why_bimanual", ""),
        components=components_to_text(components),
        role_verbs=", ".join(ROLE_VERBS),
        role_gloss=ROLE_GLOSS,
        relation_examples=", ".join(RELATION_EXAMPLES),
    ),
    ROLE_DECOMP_PROMPT["system"], model, image_url=GRID_PATH
)
decomp = parse_json(decomp_raw)
print("\nparsed decomposition:")
print(json.dumps(decomp, indent=2, ensure_ascii=False))

RAW OUTPUT:
```json
{
  "roles": [
    {
      "id": "A",
      "role": "hold",
      "target": "body",
      "contact_region": "left side of the box",
      "function": "provide a stable prehensile grip to maintain the box's position"
    },
    {
      "id": "B",
      "role": "hold",
      "target": "body",
      "contact_region": "right side of the box",
      "function": "provide a stable prehensile grip to maintain the box's position"
    }
  ],
  "relation": "balanced grip on opposite sides"
}
```

parsed decomposition:
{
  "roles": [
    {
      "id": "A",
      "role": "hold",
      "target": "body",
      "contact_region": "left side of the box",
      "function": "provide a stable prehensile grip to maintain the box's position"
    },
    {
      "id": "B",
      "role": "hold",
      "target": "body",
      "contact_region": "right side of the box",
      "function": "provide a stable prehensile grip to maintain the box's position"
    }
  ],
  "relation": "balanced grip on

In [40]:
# --- validate the decomposition (roles closed, relation open) ---
def validate_decomposition(dec, components):
    names  = {c["name"].lower() for c in components}
    roles  = dec.get("roles", [])
    issues = []
    if len(roles) != 2:
        issues.append(f"expected 2 roles, got {len(roles)}")
    for r in roles:
        if r.get("role") not in ROLE_VERBS:
            issues.append(f"role {r.get('role')!r} not in ROLE_VERBS")
        if str(r.get("target", "")).lower() not in names:
            issues.append(f"target {r.get('target')!r} is not a groundable part")
    rel = dec.get("relation")
    if not isinstance(rel, str) or not rel.strip():
        issues.append("relation is empty (should be a short free-form phrase)")
    # contact_region drives the downstream Molmo query, so the two MUST point to different places.
    if len(roles) == 2:
        regs = [str(r.get("contact_region", "")).strip().lower() for r in roles]
        if not all(regs):
            issues.append("a role is missing contact_region")
        elif regs[0] == regs[1]:
            issues.append("both roles share the same contact_region (Molmo would point to one place)")
    return issues

issues = validate_decomposition(decomp, components)
print("VALID" if not issues else "ISSUES:")
for it in issues:
    print("  -", it)

VALID


## 5 · Role pair pattern  *(NEW, deterministic)*

Pure normalization — no model call. The role pair stays hand-agnostic; left/right assignment is deferred to the robot-embodiment stage. `relation` is carried through verbatim from the model's free-form phrase.

In [46]:
# --- Step D: role pair pattern (deterministic) ---
def normalize_pair(dec):
    roles = dec.get("roles", [])
    ra, rb = roles[0]["role"], roles[1]["role"]
    return {
        "pattern":       f"{ra} + {rb}",
        "relation":      dec.get("relation"),   # free-form, passed through
        "symmetric":     ra == rb,
        "hand_agnostic": True,   # left/right assigned later, at robot-embodiment stage
    }

pattern = normalize_pair(decomp)
print(json.dumps(pattern, indent=2, ensure_ascii=False))

{
  "pattern": "hold + hold",
  "relation": "balanced grip on opposite sides",
  "symmetric": true,
  "hand_agnostic": true
}


## 6 · Assemble + (optionally) save the record

`answer` holds the two role targets — the regions Molmo will later ground into the two heatmaps. Set `SAVE = True` to append this record to `OUT_PATH`.

In [48]:
# --- Step E: assemble the final per-object dataset record (query-ready) ---
def molmo_instruction(role):
    """Richer, role-conditioned Molmo pointing instruction:
       'Point to <region> where a robot hand should <verb> it to <function>.'"""
    region = str(role.get("contact_region") or role.get("target") or "").strip().rstrip(".")
    region_phrase = region if region.lower().startswith("the ") else f"the {region}"
    verb = str(role.get("role", "")).strip()
    fn = str(role.get("function") or "").strip().rstrip(".")
    q = f"Point to {region_phrase}"
    if verb:
        q += f" where a robot hand should {verb} it"
        if fn:
            q += f" to {fn}"
    return q + "."

_roles = decomp.get("roles", [])
record = {
    **object_record,
    "task":          chosen.get("task"),                 # short label (metadata)
    "query":         chosen.get("query"),                # dataset Task query ("How would you ...?")
    "goal":          chosen.get("goal"),
    "why_bimanual":  chosen.get("why_bimanual"),
    "roles":         _roles,                             # full parsed decomposition kept as metadata
    **pattern,
    # what actually gets fed to Molmo: one role-conditioned "Point to ..." instruction per role (A, B)
    "molmo_queries": [molmo_instruction(r) for r in _roles],
    "answer":        [r.get("contact_region") or r.get("target") for r in _roles],
}
print(json.dumps(record, indent=2, ensure_ascii=False))

SAVE = False
if SAVE:
    existing = json.load(open(OUT_PATH)) if os.path.exists(OUT_PATH) else []
    existing = [r for r in existing if r.get("object_id") != record["object_id"]] + [record]
    json.dump(existing, open(OUT_PATH, "w"), indent=2, ensure_ascii=False)
    print(f"\nsaved -> {OUT_PATH}  ({len(existing)} records)")

{
  "object_id": "6222a864318d4eba97eb2d846c14d81d",
  "object_name": "Box",
  "views_used": [
    "/nfs_drive/gobjaverse/daily-used/40/211915/00000/00000.png",
    "/nfs_drive/gobjaverse/daily-used/40/211915/00010/00010.png",
    "/nfs_drive/gobjaverse/daily-used/40/211915/00020/00020.png",
    "/nfs_drive/gobjaverse/daily-used/40/211915/00030/00030.png"
  ],
  "components": [
    {
      "name": "body",
      "interaction": "The entire surface of the box can be grasped or moved."
    }
  ],
  "task": "Carry the box",
  "query": "How would you carry the box?",
  "goal": "at point A -> at point B",
  "why_bimanual": "A large box requires two hands to maintain stability and balance while moving.",
  "roles": [
    {
      "id": "A",
      "role": "hold",
      "target": "body",
      "contact_region": "left side of the box",
      "function": "provide a stable prehensile grip to maintain the box's position"
    },
    {
      "id": "B",
      "role": "hold",
      "target": "body",
    

## Once the prompts feel right

- Move `FILTER_PROMPT`, `TASK_PROPOSAL_PROMPT`, `ROLE_DECOMP_PROMPT` into `prompt/get_component_prompt.json` (replacing the old `filter` + `affordance` blocks).
- For the filter: update `get_component.py::is_operable()` to parse the JSON verdict (`{"keep": ...}`) instead of the old strict `== "yes"`.
- Rewrite `get_affordance.py`:
  - feed the grid image (`model.text_image`, **not** `model.text`) — the current step is image-blind.
  - emit `{task, goal, roles, relation, pattern, answer}` instead of `{reason, affordance_parts}`.
- Update `run_daily_used_pilot.py` / `get_question.py` to read `record["roles"]` instead of `aff["reason"]` / `aff["affordance_parts"]`.